In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import re
import os
from dotenv import load_dotenv

load_dotenv()
loan_dataset_path = os.getenv('loan_dataset_path')
df = pd.read_csv(loan_dataset_path)
df.head(10)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
5,21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,2.0,532,No,1
6,26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,3.0,701,No,1
7,24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,4.0,585,No,1
8,24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.90,0.35,2.0,544,No,1
9,21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,3.0,640,No,1


Looking at the columns, we can tell this dataset involves Binary Logistic Regression since our target, loan_status, has the option of being **accepted**(1) or **denied** (0).

First, we can check if there are any null / NaN values in the dataset:

In [3]:
print(df.isnull().sum())

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64


There are no null or NaNs so we can move to searching for outliers in the dataset using the describe() function

In [4]:
print(df.describe())

         person_age  person_income  person_emp_exp     loan_amnt  \
count  45000.000000   4.500000e+04    45000.000000  45000.000000   
mean      27.764178   8.031905e+04        5.410333   9583.157556   
std        6.045108   8.042250e+04        6.063532   6314.886691   
min       20.000000   8.000000e+03        0.000000    500.000000   
25%       24.000000   4.720400e+04        1.000000   5000.000000   
50%       26.000000   6.704800e+04        4.000000   8000.000000   
75%       30.000000   9.578925e+04        8.000000  12237.250000   
max      144.000000   7.200766e+06      125.000000  35000.000000   

       loan_int_rate  loan_percent_income  cb_person_cred_hist_length  \
count   45000.000000         45000.000000                45000.000000   
mean       11.006606             0.139725                    5.867489   
std         2.978808             0.087212                    3.879702   
min         5.420000             0.000000                    2.000000   
25%         8.590000  

Here we can tell a few outliers after comparing the max values to the 75th percentile.
* **person_age** has a max of 144 years which is definitely an outlier given that the 75th percentile is 30.
* **person_income** has a max of 7.2E+06 or 7,200,000, which is also an outlier since the 75th percentile is 9.5E+04.
* **person_emp_exp** has a max of 125 which is also an outlier since the 75th percentile is 8.

To remove these outliers, a couple actions are needed:
1. Set a threshold of 100 for person_age.
2. Set a threshold to limit extremely high values for person_income.
3. Set a threshold to limit extremely high values for person_emp_exp

We can just use a hard cap for #1 but for #2 and #3 it would be better to use the IQR method to set caps based on quartiles.

In [5]:
# Ensures person_age column can only contain values below 100
df = df.loc[df['person_age'] < 100]

# Ensures that person_income excludes extreme outliers using the IQR method. We still want to keep as much good info as possible so we
# are more lenient with the cutoff
Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3.0 * IQR
df = df.loc[df['person_income'] <= upper_bound]

# Ensures person_emp_exp excludes extreme outliers also using the IQR method.
Q1 = df['person_emp_exp'].quantile(0.25)
Q3 = df['person_emp_exp'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3.0 * IQR
df = df.loc[df['person_emp_exp'] <= upper_bound]

# Checks again for outliers
print(df.describe())

         person_age  person_income  person_emp_exp     loan_amnt  \
count  44136.000000   44136.000000    44136.000000  44136.000000   
mean      27.520595   75372.687103        5.166304   9472.724760   
std        5.415399   39805.193755        5.414249   6199.766187   
min       20.000000    8000.000000        0.000000    500.000000   
25%       24.000000   46850.000000        1.000000   5000.000000   
50%       26.000000   66833.500000        4.000000   8000.000000   
75%       30.000000   93901.500000        8.000000  12000.000000   
max       58.000000  241503.000000       29.000000  35000.000000   

       loan_int_rate  loan_percent_income  cb_person_cred_hist_length  \
count   44136.000000         44136.000000                44136.000000   
mean       10.994855             0.140949                    5.733483   
std         2.975634             0.087091                    3.625658   
min         5.420000             0.000000                    2.000000   
25%         8.590000  

Done with the outliers

Now since our group is working with Logistic Regression, we need to convert any and all columns with string values into numbers (integers, floats, etc).
Looking back at the columns from the first cell, we see 5 columns that have string values:
1. person_gender
2. person_education
3. person_home_ownership
4. loan_intent
5. previous_loan_defaults_on_file

We can get all the types of values for each of these columns:

In [6]:
print(df['person_gender'].unique(), "\n")
print(df['person_education'].unique(), "\n")
print(df['person_home_ownership'].unique(), "\n")
print(df['loan_intent'].unique(), "\n")
print(df['previous_loan_defaults_on_file'].unique(), "\n")

<StringArray>
['female', 'male']
Length: 2, dtype: str 

<StringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str 

<StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str 

<StringArray>
[         'PERSONAL',         'EDUCATION',           'MEDICAL',
           'VENTURE',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str 

<StringArray>
['No', 'Yes']
Length: 2, dtype: str 



1. person_gender: female, male
2. person_education: High School, Associate, Bachelor, Master, Doctorate
3. person_home_ownership: RENT, OWN, MORTGAGE, OTHER
4. loan_intent: PERSONAL, EDUCATION, MEDICAL, VENTURE, HOMEIMPROVEMENT, DEBTCONSOLIDATION
5. previous_loan_defaults_on_file: No, Yes

We then categorize each of these columns so that the ones with weight are not mixed with non-weighted columns:
* **person_gender**: binary mapping
* **person_education**: weighted (so we'll use mapping based on education)
* **person_home_ownership**: unweighted (dummy variable)
* **loan_intent**: unweighted (dummy variable)
* **previous_loan_defaults_on_file**: binary mapping

In [7]:
# Binary mapping for person_gender into Male (0) Female (1)
df['person_gender'] = df['person_gender'].map({'male': 0, 'female': 1})

# Weighted mapping for person_education
education = {
    "High School": 0,
    "Associate": 1,
    "Bachelor": 2,
    "Master": 3,
    "Doctorate": 4
}

df['person_education'] = df['person_education'].map(education)

# Dummy variables
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent'], drop_first=True)

# Binary mapping for previous_loan_defaults_on_file
df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].map({'No': 0, 'Yes': 1})

In [8]:
# Check to see everything is a number
df.info()

<class 'pandas.DataFrame'>
Index: 44136 entries, 0 to 44999
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      44136 non-null  float64
 1   person_gender                   44136 non-null  int64  
 2   person_education                44136 non-null  int64  
 3   person_income                   44136 non-null  float64
 4   person_emp_exp                  44136 non-null  int64  
 5   loan_amnt                       44136 non-null  float64
 6   loan_int_rate                   44136 non-null  float64
 7   loan_percent_income             44136 non-null  float64
 8   cb_person_cred_hist_length      44136 non-null  float64
 9   credit_score                    44136 non-null  int64  
 10  previous_loan_defaults_on_file  44136 non-null  int64  
 11  loan_status                     44136 non-null  int64  
 12  person_home_ownership_OTHER     44136 non-null  

Strings were converted to floats and ints and our unweighted (dummy variables) are in the table. We are done with the cleaning portion of the project


In [ ]:
load_dotenv()
clean_loan_dataset_path = os.getenv('clean_loan_dataset_path')

df.to_csv(clean_loan_dataset_path, index=False)